In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

import umap
import warnings
warnings.filterwarnings('ignore')

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [3]:
df=pd.read_csv("final_database.csv")
df.shape

(5546, 29)

In [4]:
binary_columns = ['jobma_catcher_status',
                  'jobma_catcher_type',
                  'is_premium',
                  'subscription_type_x',
                  'referral_credit',
                  'ai_live_interview',
                  'subscription_type_y',
                  'plan_type',
                  'is_unlimited',
                  'currency'
                 ]

categorical_columns = ['subscription_status',
                       'per_sub_user',
                       'company_size',
                      ]
                   
ordinal_columns = ['sub_user',#
                   'days_since_creation',#
                   'number_of_logins',#
                   'days_since_login',#
                   'wallet_sum',#
                   'subscription_sum',#
                   'subscription_count',#
                   'ai_kit_counts',#
                   'pre_recorded_kit_counts',#
                   'number_of_jobs_posted',#
                   'number_of_pitcher_invitations',#
                   'number_of_pitchers_applied',#
                   'pre_rec_interviews_counts',#
                   'live_interviews_counts',#
                   'ai_interviews_counts',#done
                   ]

In [5]:
# Clean binary values , # Remove negatives and  clamp to 0
def clean_binary(X):
    return X.apply(lambda col: col.map(lambda val: 1 if pd.to_numeric(val, errors='coerce') > 0 else 0))
    
def clean_categorical(X):
    return X.apply(lambda col: col.map(lambda val: str(val).strip().lower() if pd.notnull(val) else 'others'))

def clean_ordinal(X):
    return X.clip(lower=0)

In [6]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, encoding_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, encoding_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded, encoded

In [7]:
#pipeline 
binary_pipeline = Pipeline(steps=[('cleaner', FunctionTransformer(clean_binary, validate=False)),
                                  ('imputer', SimpleImputer(strategy='most_frequent'))
                                 ])

categorical_pipeline = Pipeline(steps=[('cleaner', FunctionTransformer(clean_categorical, validate=False)),
                                       ('imputer', SimpleImputer(strategy='most_frequent')),
                                       ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
                                       ('scaler', StandardScaler())
                                      ])

ordinal_pipeline = Pipeline(steps=[('cleaner', FunctionTransformer(clean_ordinal, validate=False)),
                                   ('imputer', SimpleImputer(strategy='median')),
                                   ('scaler', RobustScaler())
                                  ])

# Combined preprocessor
preprocessor = ColumnTransformer(transformers=[('bin', binary_pipeline, binary_columns),
                                               ('cat', categorical_pipeline, categorical_columns),
                                               ('ord', ordinal_pipeline, ordinal_columns)
                                              ])

pipeline = Pipeline([('preprocessing', preprocessor)])

In [8]:
# Training
def train_autoencoder(X_train, input_dim, encoding_dim, epochs=50, batch_size=64):
    model = Autoencoder(input_dim, encoding_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    
    X_tensor = torch.tensor(X_train).float()
    dataset = torch.utils.data.TensorDataset(X_tensor)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for batch in dataloader:
            optimizer.zero_grad()
            x_batch = batch[0]
            reconstructed, _ = model(x_batch)
            loss = criterion(reconstructed, x_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}")
    
    with torch.no_grad():
        _, embeddings = model(X_tensor)
    return model, embeddings

In [9]:
# Clustering
def perform_clustering(embeddings, n_clusters=5):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    cluster_labels = kmeans.fit_predict(embeddings)
    return kmeans, cluster_labels

In [10]:
#Visualization
def visualize_clusters(embeddings, cluster_labels, method='umap'):
    if method == 'pca':
        reducer = PCA(n_components=2)
    else:
        reducer = umap.UMAP(n_components=2, random_state=42)
    
    reduced = reducer.fit_transform(embeddings)
    
    plt.figure(figsize=(8, 6))
    plt.scatter(reduced[:, 0], reduced[:, 1], c=cluster_labels, cmap='viridis', alpha=0.7)
    plt.title(f"Cluster Visualization ({method.upper()})")
    plt.colorbar()
    plt.show()

In [11]:
#Main DEC Pipeline
def deep_clustering_pipeline(df, binary_columns, categorical_columns, ordinal_columns, n_clusters=5, encoding_dim=2):
    print("→ Preprocessing...")
    X_processed = preprocessor.fit_transform(df)

    print("→ Training Autoencoder...")
    model, embeddings = train_autoencoder(X_processed, X_processed.shape[1], encoding_dim)

    print("→ Performing Clustering...")
    kmeans, cluster_labels = perform_clustering(embeddings.detach().numpy(), n_clusters)

    print("→ Visualizing Clusters...")
    visualize_clusters(embeddings.detach().numpy(), cluster_labels, method='umap')

    sil_score = silhouette_score(embeddings.detach().numpy(), cluster_labels)
    print(f"Silhouette Score: {sil_score:.4f}")

    return model, kmeans, cluster_labels, sil_score

In [12]:
# model, kmeans, cluster_labels, sil_score = deep_clustering_pipeline(
#     df, binary_columns, categorical_columns, ordinal_columns, n_clusters=5, encoding_dim=2
# )


In [ ]:
processed_data = pipeline.fit_transform(df)
# X_processed = preprocess_data(df, binary_columns, categorical_columns, ordinal_columns)


In [ ]:
X_scaled = pipeline.fit_transform(X)
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
dataset = TensorDataset(X_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

In [ ]:
# Pretraining Autoencoder
input_dim = X_scaled.shape[1]
autoencoder = AutoEncoder(input_dim).to(device)
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
criterion = nn.SmoothL1Loss()  # More robust than MSE with 

In [ ]:
#Pretraining Autoencoder
for epoch in range(50):
    autoencoder.train()
    total_loss = 0
    for batch in dataloader:
        x_batch = batch[0].to(device)
        x_bar, _ = autoencoder(x_batch)
        loss = criterion(x_bar, x_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"[Pretrain] Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

In [ ]:
# Extract Latent Features
autoencoder.eval()
with torch.no_grad():
    _, latent = autoencoder(X_tensor.to(device))
latent_np = latent.cpu().numpy()

In [ ]:
# Determine Optimal Cluster Count   #Finding optimal number of clusters using Silhouette Score
scores = []
K_range = range(2, 15)
for k in K_range:
    kmeans = KMeans(n_clusters=k, n_init=10)
    labels = kmeans.fit_predict(latent_np)
    score = silhouette_score(latent_np, labels)
    scores.append(score)
    print(f"Clusters: {k}, Silhouette Score: {score:.4f}")

In [ ]:
# Plotting
plt.figure(figsize=(8, 5))
plt.plot(K_range, scores, marker='o', color='navy', linewidth=2)
plt.title("Silhouette Score vs. Number of Clusters")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.xticks(K_range)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#Initialising KMeans
kmeans = KMeans(n_clusters=5, n_init=20)
cluster_ids = kmeans.fit_predict(latent_np)
cluster_centers = torch.tensor(kmeans.cluster_centers_, dtype=torch.float32).to(device)

In [ ]:
# Deep Embedding Clusturing Model
class DEC(nn.Module):
    def __init__(self, encoder, cluster_centers):
        super(DEC, self).__init__()
        self.encoder = encoder
        self.cluster_centers = nn.Parameter(cluster_centers)

    def forward(self, x):
        z = self.encoder(x)
        q = 1.0 / (1.0 + torch.sum((z.unsqueeze(1) - self.cluster_centers)**2, dim=2))
        q = q.pow((1 + 1) / 2.0)
        q = (q.t() / torch.sum(q, dim=1)).t()
        return q, z

def target_distribution(q):
    weight = q**2 / q.sum(0)
    return (weight.t() / weight.sum(1)).t()  

#Train DEC
dec = DEC(autoencoder.encoder, cluster_centers.clone()).to(device)
optimizer = torch.optim.Adam(dec.parameters(), lr=1e-3)

In [ ]:
#Training DEC (with reconstruction loss)

In [ ]:
for epoch in range(50):
    dec.train()
    total_loss = 0
    all_q = []

    for batch in dataloader:
        x = batch[0].to(device)
        q, _ = dec(x)
        all_q.append(q.detach().cpu())
    q_all = torch.cat(all_q, dim=0)
    p_all = target_distribution(q_all)

    entropy = -torch.sum(q_all * torch.log(q_all + 1e-10)) / q_all.size(0)

    idx = 0
    for batch in dataloader:
        x = batch[0].to(device)
        q, z = dec(x)
        p = p_all[idx:idx + len(x)].to(device)
        kl_loss = F.kl_div(q.log(), p, reduction='batchmean')
        x_bar = autoencoder.decoder(z)
        recon_loss = F.smooth_l1_loss(x_bar, x)
        loss = kl_loss + 0.1 * recon_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        idx += len(x)
        total_loss += loss.item()

    print(f"[DEC] Epoch {epoch+1}, KL Loss: {total_loss/len(dataloader):.4f}, Entropy: {entropy:.4f}")

In [ ]:
#Final Prediction
dec.eval()
with torch.no_grad():
    q, _ = dec(X_tensor.to(device))
    preds = torch.argmax(q, dim=1).cpu().numpy()

df['dec_cluster'] = preds
print("Clustering complete")

In [ ]:
df['dec_cluster'].head(10)

In [ ]:
#Visualizing clusters with t-SNE..

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
latent_2d = tsne.fit_transform(latent_np)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=latent_2d[:, 0], y=latent_2d[:, 1], hue=preds, palette='tab10', s=30, edgecolor='k')
plt.title("Clustered Latent Space (t-SNE)")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
sil_score = silhouette_score(latent_np, preds)
print(f"Silhouette Score: {sil_score}")

In [ ]:
# Checking the size of each cluster
cluster_sizes = pd.Series(preds).value_counts()
cluster_sizes

In [ ]:
# Visualize the cluster distribution
plt.figure(figsize=(10, 5))
cluster_sizes.plot(kind='bar', color='lightblue', edgecolor='black')
plt.title("Cluster Size Distribution")
plt.xlabel("Cluster")
plt.ylabel("Number of Data Points")
plt.grid(True)
plt.tight_layout()
plt.show()